# 入门案例：从最小拓扑到网络流仿真

本文件给出可以直接运行和改造的 Ether 入门案例。

## 案例 1：创建两个节点

In [3]:
from ether.core import Node, Capacity

a = Node(
    "edge-a",
    capacity=Capacity(cpu_millis=4000, memory=4 * 1024 * 1024 * 1024),
    arch="x86",
    labels={"role": "edge"}
)

b = Node(
    "edge-b",
    capacity=Capacity(cpu_millis=4000, memory=4 * 1024 * 1024 * 1024),
    arch="x86",
    labels={"role": "edge"}
)

print(a.name, a.capacity, a.arch, a.labels)

edge-a Capacity(CPU: 4000 Memory: 4294967296) x86 {'role': 'edge'}



你会得到两个计算节点。此时它们还没有网络连接。

## 案例 2：创建一条可传输的最小拓扑

In [4]:
from ether.core import Node, Link, Connection
from ether.topology import Topology

topology = Topology()

a = Node("a")
b = Node("b")

link_a = Link(bandwidth=100, tags={"name": "a-link"})
link_b = Link(bandwidth=100, tags={"name": "b-link"})
switch = "switch"

topology.add_connection(Connection(a, link_a, latency=1))
topology.add_connection(Connection(link_a, switch, latency=1))
topology.add_connection(Connection(switch, link_b, latency=1))
topology.add_connection(Connection(link_b, b, latency=1))

route = topology.route(a, b)

print("path:", route.path)
print("hops:", route.hops)
print("rtt:", route.rtt)

path: [a, Link(0x26f5e507fa0){'name': 'a-link'}, 'switch', Link(0x26f1ef3f370){'name': 'b-link'}, b]
hops: [Link(0x26f5e507fa0){'name': 'a-link'}, Link(0x26f1ef3f370){'name': 'b-link'}]
rtt: 8


这里的关键是：

- `a` 和 `b` 没有直接相连。
- 两端各有一条 `Link`。
- 中间有一个透明 switch。
- `route.hops` 中只有 `Link`。

## 案例 3：模拟一次网络传输

In [5]:
import simpy

from ether.core import Node, Link, Connection, Flow
from ether.topology import Topology

env = simpy.Environment()
topology = Topology()

a = Node("a")
b = Node("b")
link_a = Link(bandwidth=100, tags={"name": "a-link"})
link_b = Link(bandwidth=100, tags={"name": "b-link"})
switch = "switch"

topology.add_connection(Connection(a, link_a, latency=1))
topology.add_connection(Connection(link_a, switch, latency=1))
topology.add_connection(Connection(switch, link_b, latency=1))
topology.add_connection(Connection(link_b, b, latency=1))

route = topology.route(a, b)

size = 10 * 1024 * 1024
flow = Flow(env, size=size, route=route)

flow.start()
env.run()

print("finished at", env.now)

finished at 0.8768049484536082


这个案例会模拟 10 MiB 数据从 `a` 传到 `b`。

传输时间大致由以下因素决定：

- 路由 RTT。
- 路由中每条 Link 的带宽。
- TCP goodput 近似系数。

## 案例 4：模拟共享瓶颈链路

这个案例展示两个 Flow 同时经过同一条 bottleneck 链路。

In [6]:
import simpy

from ether.core import Node, Link, Connection, Flow
from ether.topology import Topology

env = simpy.Environment()
topology = Topology()

a = Node("a")
b = Node("b")
c = Node("c")

link_a = Link(100, tags={"name": "a-link"})
link_c = Link(100, tags={"name": "c-link"})
bottleneck = Link(10, tags={"name": "bottleneck"})
link_b = Link(100, tags={"name": "b-link"})

sw_left = "left-switch"
sw_right = "right-switch"

topology.add_connection(Connection(a, link_a))
topology.add_connection(Connection(link_a, sw_left))

topology.add_connection(Connection(c, link_c))
topology.add_connection(Connection(link_c, sw_left))

topology.add_connection(Connection(sw_left, bottleneck))
topology.add_connection(Connection(bottleneck, sw_right))
topology.add_connection(Connection(sw_right, link_b))
topology.add_connection(Connection(link_b, b))

route_a_b = topology.route(a, b)
route_c_b = topology.route(c, b)

size = 10 * 1024 * 1024

f1 = Flow(env, size, route_a_b)
f2 = Flow(env, size, route_c_b)

f1.start()
f2.start()
env.run()

print("finished at", env.now)

finished at 17.296098969072165


两个流都会经过 `bottleneck`。当第二个流加入时，Ether 会触发带宽重分配，并中断已经在传输的流，让它重新计算剩余传输时间。

## 案例 5：使用 LANCell 快速创建局域网

In [7]:
from ether.cell import LANCell
from ether.blocks import nodes

lan = LANCell([
    nodes.rpi3,
    nodes.nuc,
    nodes.tx2
])

topology = lan.generate()

print("nodes:", topology.get_nodes())
print("links:", topology.get_links())

nodes: [rpi3_0, nuc_0, tx2_0]
links: [Link(0x26f5ec71120){'name': 'link_rpi3_0', 'type': 'node'}, Link(0x26f5ec70c10){'name': 'link_nuc_0', 'type': 'node'}, Link(0x26f5ec71210){'name': 'link_tx2_0', 'type': 'node'}]


这里不需要手写 `Connection`，`LANCell` 会自动创建内部 switch 和 Host 接入链路。

## 案例 6：使用 SharedLinkCell 创建共享出口

In [8]:
from ether.cell import SharedLinkCell
from ether.blocks import nodes
from ether.blocks.cells import MobileConnection

cell = SharedLinkCell(
    nodes=[nodes.rpi3, nodes.rpi3, nodes.nuc],
    shared_bandwidth=50,
    backhaul=MobileConnection("internet")
)

topology = cell.generate()

print("nodes:", topology.get_nodes())
for link in topology.get_links():
    print(link)


nodes: [rpi3_1, rpi3_2, nuc_1]
Link(0x26f5ec47100){'name': 'link_rpi3_1', 'type': 'node'}
Link(0x26f5ec44af0){'name': 'shared_0', 'type': 'shared'}
Link(0x26f5ec44d00){'name': 'link_rpi3_2', 'type': 'node'}
Link(0x26f5ec44f70){'name': 'link_nuc_1', 'type': 'node'}
Link(0x26f5ec47f70){'type': 'uplink', 'name': 'up_shared_0'}
Link(0x26f5ec450c0){'type': 'downlink', 'name': 'down_shared_0'}


这个拓扑适合模拟多个边缘设备共享一条移动网络出口。

## 案例 7：使用预置场景

In [9]:
from ether.topology import Topology
from ether.scenarios.urbansensing import UrbanSensingScenario

topology = Topology()
topology.add(UrbanSensingScenario(num_cells=2))

print("nodes:", len(topology.get_nodes()))
print("links:", len(topology.get_links()))

nodes: 32
links: 40



预置场景适合快速构造更复杂的实验网络。

## 常见错误

### 1. Node 直接连接 Node

错误：

```python
topology.add_connection(Connection(a, b))
```

会报错，因为 Ether 不允许 Node 直连 Node。

正确做法：

```python
topology.add_connection(Connection(a, link))
topology.add_connection(Connection(link, b))
```

更推荐：

```python
Host(a)
LANCell([...])
SharedLinkCell([...])
```

### 2. route.hops 为空

如果一条路径里没有 `Link`，`Flow` 会无法传输。

检查方式：

```python
route = topology.route(a, b)
print(route.path)
print(route.hops)
```

确保 `route.hops` 至少有一个 `Link`。

### 3. 忘记运行 env.run()

`Flow.start()` 只是把传输注册到 SimPy 事件队列，真正推进时间需要：

```python
env.run()
```